# GSM8956239 · Xenium Single-Sample QC

**Data:** GSE295730 / GSM8956239  
**Environment:** spatial (py3.10) · scanpy 1.11.5 · squidpy 1.6.5  
**Before running:** Launch on a compute node (`srun --mem=16G ...`), not on head node

In [1]:
from pathlib import Path

SAMPLE_ID = "GSM8956239"
DATA      = Path("/network/rit/lab/meng_lab/Public_data/GSE295730/extracted/GSM8956239")
WORK      = Path("/network/rit/lab/meng_lab/YS/workfolder")
OUT       = WORK / "results" / f"{SAMPLE_ID}_qc_v1"

if OUT.exists():
    raise FileExistsError(f"Output directory already exists, use v2: {OUT}")
OUT.mkdir(parents=True)
print(f"Output directory: {OUT}")

FileExistsError: Output directory already exists, use v2: /network/rit/lab/meng_lab/YS/workfolder/results/GSM8956239_qc_v1

## 1. Experiment Metadata

In [2]:
import json

exp = json.loads((DATA / "experiment.xenium").read_text())
for k, v in exp.items():
    print(f"{k:35s}: {v}")
# Check instrument_type and image_channel_names to confirm image type

major_version                      : 5
minor_version                      : 0
patch_version                      : 0
run_name                           : SARAH_STANLEY
run_start_time                     : 2024-09-03T17:55:40Z
region_name                        : v2
preservation_method                : ffpe
num_cells                          : 68273
transcripts_per_cell               : 352
transcripts_per_100um              : 295.4073051142696
region_area                        : 31038067.698124997
total_cell_area                    : 11195583.327638716
thickness_of_high_quality_decoded_transcripts: 4.014712805706926
nuclear_transcripts_per_100um      : 539.0816953373999
fraction_transcripts_assigned      : 0.6982531158046691
segmented_cell_stain_frac          : 0.0
segmented_cell_boundary_frac       : 0.0
segmented_cell_interior_frac       : 0.0
segmented_cell_nuc_expansion_frac  : 1.0
imported_cell_frac                 : 0.0
cassette_name                      : REP1
slide_id          

## 2. Official QC Metrics

In [3]:
import pandas as pd

metrics = pd.read_csv(DATA / "metrics_summary.csv").T
metrics.columns = ["Value"]
metrics

,Value
run_name,SARAH_STANLEY
cassette_name,REP1
region_name,v2
panel_name,mBrain_100g
panel_design_id,C728YM
predesigned_panel_id,mBrain_v1.1
region_area,31038067.698125
total_cell_area,11195583.327639
total_high_quality_decoded_transcripts,47364731
fraction_transcripts_decoded_q20,0.91509


## 3. Load Data (scanpy)

In [4]:
import scanpy as sc
import numpy as np

sc.settings.verbosity = 2

adata = sc.read_10x_h5(DATA / "cell_feature_matrix.h5")
print(adata)
print(f"\nCells:    {adata.n_obs:,}")
print(f"Features: {adata.n_vars:,}")
print(f"\nFeature types:")
print(adata.var["feature_types"].value_counts())

reading /network/rit/lab/meng_lab/Public_data/GSE295730/extracted/GSM8956239/cell_feature_matrix.h5
 (0:00:00)
AnnData object with n_obs × n_vars = 68273 × 347
    var: 'gene_ids', 'feature_types', 'genome'

Cells:    68,273
Features: 347

Feature types:
feature_types
Gene Expression    347
Name: count, dtype: int64


In [5]:
# Keep Gene Expression features only
adata = adata[:, adata.var["feature_types"] == "Gene Expression"].copy()
print(f"Gene Expression features: {adata.n_vars}")
print(f"Panel gene examples: {list(adata.var_names[:10])}")

Gene Expression features: 347
Panel gene examples: ['2010300C02Rik', 'Acsbg1', 'Acta2', 'Acvrl1', 'Adamts2', 'Adamtsl1', 'Adcyap1', 'Adgrl4', 'Aldh1a2', 'Angpt1']


In [6]:
# Add spatial coordinates
cells = pd.read_parquet(DATA / "cells.parquet")
cells["cell_id"] = cells["cell_id"].astype(str)
cells = cells.set_index("cell_id")

# Align to matrix order
adata.obs = adata.obs.join(cells[["x_centroid", "y_centroid"]], how="left")
adata.obsm["spatial"] = adata.obs[["x_centroid", "y_centroid"]].to_numpy()

print(f"Coordinate range:")
print(f"  x: {adata.obs['x_centroid'].min():.1f} ~ {adata.obs['x_centroid'].max():.1f} um")
print(f"  y: {adata.obs['y_centroid'].min():.1f} ~ {adata.obs['y_centroid'].max():.1f} um")

Coordinate range:
  x: 5.0 ~ 8445.9 um
  y: 5.7 ~ 4352.2 um


## 4. QC Metrics

In [7]:
sc.pp.calculate_qc_metrics(adata, percent_top=None, inplace=True)

print(f"Median transcripts per cell:    {adata.obs['total_counts'].median():.0f}")
print(f"Median genes detected per cell: {adata.obs['n_genes_by_counts'].median():.0f}")
print(f"Cells with 0 transcripts:       {(adata.obs['total_counts'] == 0).sum():,}")

adata.obs[["total_counts", "n_genes_by_counts"]].describe()

Median transcripts per cell:    351
Median genes detected per cell: 94
Cells with 0 transcripts:       201


,total_counts,n_genes_by_counts
count,68273.000000,68273.000000
mean,484.416534,93.781905
std,380.139832,37.156028
min,0.000000,0.000000
25%,186.000000,64.000000
50%,351.000000,94.000000
75%,716.000000,125.000000
max,2715.000000,193.000000


## 5. QC Distributions

In [8]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)

# Transcript distribution
axes[0].hist(np.log10(adata.obs["total_counts"] + 1), bins=50, color="#24758B", edgecolor="none")
axes[0].axvline(np.log10(adata.obs["total_counts"].median() + 1), color="tomato", ls="--",
                label=f"median={adata.obs['total_counts'].median():.0f}")
axes[0].set(xlabel="log10(transcripts + 1)", ylabel="Cells", title="Transcript Distribution")
axes[0].legend(fontsize=8)

# Gene count distribution
axes[1].hist(adata.obs["n_genes_by_counts"], bins=50, color="#665A9B", edgecolor="none")
axes[1].axvline(adata.obs["n_genes_by_counts"].median(), color="tomato", ls="--",
                label=f"median={adata.obs['n_genes_by_counts'].median():.0f}")
axes[1].set(xlabel="Genes detected", ylabel="Cells", title="Gene Count Distribution")
axes[1].legend(fontsize=8)

# Transcripts vs genes scatter
axes[2].scatter(adata.obs["total_counts"], adata.obs["n_genes_by_counts"],
                s=0.5, alpha=0.3, color="#24758B", rasterized=True)
axes[2].set(xlabel="Total transcripts", ylabel="Genes detected", title="Transcripts vs Genes")

fig.suptitle(f"{SAMPLE_ID} - QC Distributions")
fig.savefig(OUT / "qc_distributions.png", dpi=180)
plt.show()
print(f"Saved -> {OUT}/qc_distributions.png")

Saved -> /network/rit/lab/meng_lab/YS/workfolder/results/GSM8956239_qc_v1/qc_distributions.png


## 6. Spatial Scatter Plot

In [9]:
import matplotlib.pyplot as plt

xy = adata.obsm["spatial"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

sc1 = axes[0].scatter(xy[:, 0], xy[:, 1],
                      c=adata.obs["total_counts"],
                      s=1.2, linewidths=0, cmap="viridis", rasterized=True)
axes[0].set(title="Total Transcripts", xlabel="X (um)", ylabel="Y (um)")
axes[0].set_aspect("equal")
axes[0].invert_yaxis()
fig.colorbar(sc1, ax=axes[0], label="total counts")

sc2 = axes[1].scatter(xy[:, 0], xy[:, 1],
                      c=adata.obs["n_genes_by_counts"],
                      s=1.2, linewidths=0, cmap="plasma", rasterized=True)
axes[1].set(title="Genes Detected", xlabel="X (um)", ylabel="Y (um)")
axes[1].set_aspect("equal")
axes[1].invert_yaxis()
fig.colorbar(sc2, ax=axes[1], label="n genes")

fig.suptitle(f"{SAMPLE_ID} - Spatial QC")
fig.savefig(OUT / "spatial_qc.png", dpi=180)
plt.show()
print(f"Saved -> {OUT}/spatial_qc.png")

Saved -> /network/rit/lab/meng_lab/YS/workfolder/results/GSM8956239_qc_v1/spatial_qc.png


## 7. Morphology Image Metadata

In [10]:
import tifffile

with tifffile.TiffFile(DATA / "morphology.ome.tif") as tif:
    p0 = tif.pages[0]
    print(f"Total pages (channels): {len(tif.pages)}")
    print(f"Page 0 shape (H x W):   {p0.shape}")
    print(f"Data type:              {p0.dtype}")
    if tif.ome_metadata:
        # Find Channel Name to confirm DAPI
        import re
        channels = re.findall(r'Name="([^"]+)"', tif.ome_metadata)
        print(f"Channel names:          {channels}")
        # Physical pixel size
        px = re.findall(r'PhysicalSize[XY]="([^"]+)"', tif.ome_metadata)
        units = re.findall(r'PhysicalSize[XY]Unit="([^"]+)"', tif.ome_metadata)
        print(f"Physical pixel size:    {px} {units}")

Total pages (channels): 12
Page 0 shape (H x W):   (20508, 39782)
Data type:              uint16
Channel names:          ['DAPI']
Physical pixel size:    ['0.2125', '0.2125'] ['µm', 'µm']


## 8. Save AnnData + QC Summary

In [11]:
# Save cell_qc.tsv (for the lab report sheet)
qc_cols = ["x_centroid", "y_centroid", "total_counts", "n_genes_by_counts"]
if "cell_area" in adata.obs.columns:
    qc_cols.append("cell_area")
adata.obs[qc_cols].to_csv(OUT / "cell_qc.tsv", sep="\t")

# Save AnnData for downstream modeling
adata.write_h5ad(OUT / f"{SAMPLE_ID}_qc.h5ad")

# QC summary JSON
summary = {
    "sample_id": SAMPLE_ID,
    "cells": int(adata.n_obs),
    "gene_expression_panel_features": int(adata.n_vars),
    "total_gene_transcripts": int(adata.obs["total_counts"].sum()),
    "median_transcripts_per_cell": float(adata.obs["total_counts"].median()),
    "median_genes_per_cell": float(adata.obs["n_genes_by_counts"].median()),
    "zero_count_cells": int((adata.obs["total_counts"] == 0).sum()),
    "notes": [
        "Xenium targeted panel — not whole transcriptome.",
        "No cells filtered; no cell types assigned.",
        "GSE295730 and GSE260687 are separate experiments.",
    ]
}
(OUT / "qc_summary.json").write_text(json.dumps(summary, indent=2))

print(json.dumps(summary, indent=2))
print(f"\nAll outputs -> {OUT}")

{
  "sample_id": "GSM8956239",
  "cells": 68273,
  "gene_expression_panel_features": 347,
  "total_gene_transcripts": 33072570,
  "median_transcripts_per_cell": 351.0,
  "median_genes_per_cell": 94.0,
  "zero_count_cells": 201,
  "notes": [
    "Xenium targeted panel \u2014 not whole transcriptome.",
    "No cells filtered; no cell types assigned.",
    "GSE295730 and GSE260687 are separate experiments."
  ]
}

All outputs -> /network/rit/lab/meng_lab/YS/workfolder/results/GSM8956239_qc_v1
